# IonQ Trapped-Ion Exploration

Meet IonQ's native gate set and the all-to-all connectivity that makes trapped-ion hardware special, built and verified entirely in NumPy.

**Objectives:**
- Define IonQ's native gates GPi(phi), GPi2(phi), and the Molmer-Sorensen (MS) entangler as numpy matrices
- Verify their key properties: GPi(0) = X, unitarity, GPi2 as a square root of a pi-rotation
- Show the MS gate is unitary and maximally entangling via a reduced-state purity check
- Compile a standard gate (Hadamard) exactly into the native set
- See why all-to-all connectivity means any qubit pair entangles directly, with no SWAP overhead

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.
<!-- browser-runnable -->

In [ ]:
# Setup: Run this cell first
# Requires: pip install -e '.[dev]' from the project root (see `make setup`)

from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt

# Seed any randomness so CI is deterministic (we lean on exact matrices, not samples).
np.random.seed(0)

# The local simulator is our free, instant default. IonQ hardware would charge
# $0.30 per task + $0.01 per shot -- we do not submit anything here.
device = LocalSimulator()

# Reference single-qubit matrices we will compare against.
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
H = (1 / np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)

print("Setup complete. LocalSimulator ready; native gates defined below in numpy.")

## 1. Why IonQ has a different native gate set

A trapped-ion machine encodes each qubit in the energy levels of a single charged atom and drives gates with laser pulses. The pulses do not naturally implement the textbook H / S / T / CNOT set. Instead IonQ exposes three **native** gates -- the operations the laser can perform directly:

- **GPi(phi)** -- a single-qubit pi-rotation about an axis in the equatorial plane chosen by `phi`. It is a tunable bit-flip: a generalized Pauli-X.
- **GPi2(phi)** -- a single-qubit pi/2-rotation about that same axis. It is, up to a global phase, a square root of GPi.
- **MS (Molmer-Sorensen)** -- the two-qubit entangler, driven by addressing two ions with a shared laser field. The maximally-entangling MS is $\exp(-i\,\tfrac{\pi}{4}\, X\otimes X)$.

Everything you write in Braket -- H, CNOT, RX -- gets **compiled** down to these three before it runs on the ion trap. The qcsim simulator in this lab does not implement GPi/GPi2/MS as gates, so we build them as numpy matrices and verify every claim numerically. We only assert what we can prove exactly.

$$
\mathrm{GPi}(\phi) = \begin{pmatrix} 0 & e^{-i\phi} \\ e^{i\phi} & 0 \end{pmatrix},
\qquad
\mathrm{GPi2}(\phi) = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & -i\,e^{-i\phi} \\ -i\,e^{i\phi} & 1 \end{pmatrix}.
$$

In [ ]:
def GPi(phi):
    """IonQ native pi-rotation about the axis at angle phi in the X-Y plane."""
    return np.array([[0, np.exp(-1j * phi)],
                     [np.exp(1j * phi), 0]], dtype=complex)


def GPi2(phi):
    """IonQ native pi/2-rotation about the axis at angle phi in the X-Y plane."""
    return (1 / np.sqrt(2)) * np.array([[1, -1j * np.exp(-1j * phi)],
                                        [-1j * np.exp(1j * phi), 1]], dtype=complex)


print("GPi(0) =")
print(np.round(GPi(0), 4))
print("\nGPi2(0) =")
print(np.round(GPi2(0), 4))

# Claim 1: at phi = 0, GPi is exactly the Pauli-X bit flip. (exp(0) = 1.)
assert np.allclose(GPi(0), X), "GPi(0) should equal Pauli-X"
print("\nVerified: GPi(0) == Pauli-X exactly.")

## 2. The native single-qubit gates are unitary

Every physical quantum gate must be unitary -- $M\,M^{\dagger} = I$ -- so that probability is conserved. This is not optional: a non-unitary "gate" could not be realized by any closed quantum evolution. We check it for both native gates at several angles `phi`, since the axis choice must never break unitarity. We compare against the identity with `np.allclose`, using the exact matrices rather than any sampled estimate.

In [ ]:
angles = [0.0, 0.7, 2.1, -1.3, np.pi]

for phi in angles:
    g1 = GPi(phi)
    g2 = GPi2(phi)
    assert np.allclose(g1 @ g1.conj().T, I2), f"GPi not unitary at phi={phi}"
    assert np.allclose(g2 @ g2.conj().T, I2), f"GPi2 not unitary at phi={phi}"
    print(f"phi = {phi:+.3f}:  GPi unitary = True,  GPi2 unitary = True")

print("\nVerified: GPi and GPi2 are unitary for every axis angle tested.")

## 3. GPi2 is a square root of a pi-rotation

If GPi is a pi-rotation, GPi2 is the half-step: a pi/2-rotation. Applying GPi2 twice should land you back on the full pi-rotation -- up to a global phase, which is physically irrelevant (it never affects a measurement).

Concretely, the identity is exact:

$$
\mathrm{GPi2}(\phi)^2 \;=\; -i\,\mathrm{GPi}(\phi)\qquad\text{for every }\phi.
$$

The global phase is exactly $-i$. As the simplest case, $\mathrm{GPi2}(0)^2 = -i\,X$, so squaring the native pi/2 gate is proportional to the bit-flip X. We verify the full statement across several angles, and the $-iX$ special case explicitly.

In [ ]:
for phi in angles:
    M = GPi2(phi) @ GPi2(phi)
    # Exact, for every phi: GPi2(phi)^2 = (-i) * GPi(phi).
    assert np.allclose(M, -1j * GPi(phi)), f"GPi2^2 != -i*GPi at phi={phi}"
    print(f"phi = {phi:+.3f}:  GPi2(phi)^2 == -i * GPi(phi)  -> True")

# Special case spelled out: GPi2(0) squared is proportional to X (proportionality -i).
M0 = GPi2(0) @ GPi2(0)
print("\nGPi2(0) @ GPi2(0) =")
print(np.round(M0, 4))
assert np.allclose(M0, -1j * X), "GPi2(0)^2 should equal -i * X"
print("\nVerified: GPi2(0)^2 = -i * X  (a square root of a pi-rotation, up to global phase).")

## 4. The Molmer-Sorensen gate: unitary and maximally entangling

The MS gate is the heart of trapped-ion computing -- the only two-qubit primitive the laser drives directly. The maximally-entangling form is

$$
\mathrm{MS} = \exp\!\big(-i\,\tfrac{\pi}{4}\, X\otimes X\big).
$$

Because $(X\otimes X)^2 = I$, the matrix exponential collapses to a clean closed form:

$$
\exp(-i\,\theta\, XX) = \cos\theta\, I - i\sin\theta\, XX,
\qquad \theta = \tfrac{\pi}{4}.
$$

We build it that way (no scipy needed), confirm it is unitary, then apply it to $|00\rangle$ and measure how entangled the result is.

In [ ]:
XX = np.kron(X, X)            # X tensor X, a 4x4 matrix
theta = np.pi / 4

# (XX)^2 = I, so exp(-i*theta*XX) = cos(theta)*I - i*sin(theta)*XX exactly.
MS = np.cos(theta) * np.eye(4, dtype=complex) - 1j * np.sin(theta) * XX

print("MS =")
print(np.round(MS, 4))

assert np.allclose(MS @ MS.conj().T, np.eye(4)), "MS must be unitary"
print("\nVerified: MS is unitary (MS @ MS^dagger == I).")

In [ ]:
# Apply MS to |00>, built as a state vector (big-endian: index = 2*q0 + q1).
psi00 = np.zeros(4, dtype=complex)
psi00[0] = 1.0

psi = MS @ psi00
print("MS|00> =", np.round(psi, 4))
print("        = (|00> - i|11>) / sqrt(2)")

# Reduced density matrix of qubit 0: trace out qubit 1.
rho = np.outer(psi, psi.conj()).reshape(2, 2, 2, 2)
rho0 = np.trace(rho, axis1=1, axis2=3)
purity = np.real(np.trace(rho0 @ rho0))

print("\nReduced state of qubit 0:")
print(np.round(rho0, 4))
print(f"\npurity tr(rho0^2) = {purity:.4f}")

# A product (unentangled) state would give purity 1. Maximal entanglement gives 0.5.
assert purity < 1.0 - 1e-9, "MS|00> should not be a product state"
assert np.isclose(purity, 0.5), "MS|00> should be maximally entangled (purity 0.5)"
print("\nVerified: MS|00> is entangled (purity 0.5 < 1) -- in fact maximally entangled.")

## 5. A standard gate compiles into the native set

You never hand-write GPi/GPi2/MS. You write H, CNOT, RX -- and Braket's compiler rewrites them in the native set before the ions ever see a pulse. To make that concrete, here is the Hadamard expressed exactly with two native single-qubit gates:

$$
H \;=\; \mathrm{GPi}(0)\,\cdot\,\mathrm{GPi2}(\tfrac{\pi}{2}).
$$

Reading right-to-left (circuit order): first apply GPi2(pi/2), then GPi(0). The product equals H with **zero** residual -- not approximately, exactly. This is the kind of decomposition the compiler finds for every gate in your circuit.

In [ ]:
H_native = GPi(0) @ GPi2(np.pi / 2)

print("GPi(0) @ GPi2(pi/2) =")
print(np.round(H_native, 4))
print("\nTextbook H =")
print(np.round(H, 4))

residual = np.max(np.abs(H_native - H))
print(f"\nmax |difference| = {residual:.2e}")
assert np.allclose(H_native, H), "native decomposition should equal H exactly"
print("Verified: H compiles exactly into GPi(0) . GPi2(pi/2).")

## 6. All-to-all connectivity: any pair entangles directly

The trapped-ion superpower is **all-to-all connectivity**. Every ion shares the same vibrational bus, so the MS gate can entangle *any* pair directly -- there is no notion of "adjacent" qubits and therefore no SWAP tax.

We cannot call MS on the qcsim simulator, but we can demonstrate the consequence with the standard gates qcsim does support: a Bell pair is `h(0)` then `cnot(0, 1)`, where the compiler would realize that CNOT through a single MS plus local rotations. On an ion trap this works for *any* two qubits with the same one-gate cost. The next notebook (`03-iqm-exploration`) shows the opposite world -- a nearest-neighbor lattice where distant pairs pay a chain of SWAPs first.

We verify the Bell statistics: only `00` and `11` ever appear, the signature of perfect two-qubit correlation.

In [ ]:
# Bell pair via standard gates -- the compiler would map this to one MS on real IonQ.
bell = Circuit().h(0).cnot(0, 1)
print(bell)

# Exact state vector first (no sampling): amplitudes on |01> and |10> are exactly 0.
sv = bell.state_vector()
print("\nstate vector [00, 01, 10, 11] =", np.round(sv, 4))
assert np.isclose(abs(sv[1]), 0.0) and np.isclose(abs(sv[2]), 0.0), "01 and 10 must be empty"

# Now sample. Because those amplitudes are exactly zero, every shot lands on 00 or 11.
result = device.run(bell, shots=2000).result()
counts = result.measurement_counts
print("\nmeasurement counts:", dict(counts))

observed = set(counts.keys())
assert observed.issubset({"00", "11"}), f"unexpected outcomes: {observed - {'00', '11'}}"
print("\nVerified: only 00 and 11 appear -- a directly-entangled Bell pair, no SWAP needed.")

In [ ]:
# Visualize the all-or-nothing correlation.
labels = sorted(counts.keys())
values = [counts[k] for k in labels]

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(labels, values, color="#232f3e")
ax.set_xlabel("measured bitstring (q0 q1)")
ax.set_ylabel("counts")
ax.set_title("IonQ-style Bell pair: only |00> and |11>")
for x, v in zip(labels, values):
    ax.text(x, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.show()

## Exercises

Work these in NumPy (the native gates are matrices here, not qcsim gates).

In [ ]:
# Exercise 1: GPi at a general angle is still a pi-rotation (a bit flip on the
# Bloch sphere), so its square should be the identity up to a global phase.
# Show GPi(phi) @ GPi(phi) is proportional to the identity for several phi,
# and find the proportionality constant (the global phase).
#
# TODO: Your code here
# for phi in [0.3, 1.1, -2.0]:
#     M = GPi(phi) @ GPi(phi)
#     # compare M to (phase) * I2 ; what is `phase`?
#     ...


In [ ]:
# Exercise 2: The general MS gate is exp(-i*theta*XX) for any angle theta, not just
# pi/4. Build MS(theta) = cos(theta)*I - i*sin(theta)*XX, apply it to |00>, and
# compute the reduced-qubit purity as a function of theta. At which theta is the
# state a product state (purity 1)? At which is it maximally entangled (purity 0.5)?
#
# TODO: Your code here
# def MS_gate(theta):
#     return np.cos(theta) * np.eye(4, dtype=complex) - 1j * np.sin(theta) * np.kron(X, X)
# for theta in np.linspace(0, np.pi / 2, 5):
#     ...


In [ ]:
# Exercise 3: Find a native decomposition of the S gate ([[1,0],[0,i]]) or the
# RX(theta) rotation using GPi/GPi2 (and a global phase if needed). Verify your
# decomposition matches the target matrix with np.allclose. Hint: GPi2 is a
# pi/2 rotation about an equatorial axis -- chaining two with different phi angles
# gives you a lot of reach.
#
# TODO: Your code here


## Summary

- **Native gates**: IonQ exposes only `GPi(phi)`, `GPi2(phi)`, and the `MS` entangler; every standard gate is compiled into them before running.
- **GPi(0) = X**: GPi is a tunable pi-rotation -- a generalized bit flip -- and both single-qubit native gates are unitary at every axis angle.
- **GPi2 is a square root**: `GPi2(phi)^2 = -i * GPi(phi)` exactly, so two pi/2 native gates make a pi-rotation up to a global phase.
- **MS entangles**: `MS = exp(-i*(pi/4)*XX)` is unitary and turns `|00>` into a maximally entangled state (reduced purity 0.5 < 1).
- **Exact compilation**: `H = GPi(0) . GPi2(pi/2)` to machine precision -- a concrete instance of what the compiler does for your whole circuit.
- **All-to-all connectivity**: any pair entangles with one MS, no SWAP tax. That is the trade-off that makes trapped ions shine on densely connected problems -- and the contrast the next notebook draws against a nearest-neighbor lattice.

**Next:** [03-iqm-exploration.ipynb](03-iqm-exploration.ipynb) -- superconducting qubits, nearest-neighbor topology, and the SWAP tax that all-to-all connectivity avoids.